In [27]:
from pathlib import Path
import torch
import gc

root = Path(
    "/workspace/lustre/work/pdl18409/shared/data/anomaly-detection/"
    "teams/the-outliers/DD/step3_DD_tensors/"
    "HBL_DD_tensors"
)

cavity = "010RFC"
digitizers = [101, 201, 301, 401]

files = [
    root / f"tensor_HBL-{cavity}_RFS-DIG-{digitizer}.pt"
    for digitizer in digitizers
]

for path in files:
    print(path.name, path.exists())

tensor_HBL-010RFC_RFS-DIG-101.pt True
tensor_HBL-010RFC_RFS-DIG-201.pt True
tensor_HBL-010RFC_RFS-DIG-301.pt True
tensor_HBL-010RFC_RFS-DIG-401.pt True


In [29]:
for path in files:
    obj = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print(
        path.name,
        "data =", obj["data"].shape,
        "cycle_id =", len(obj["cycle_id"])
    )

    del obj

tensor_HBL-010RFC_RFS-DIG-101.pt data = torch.Size([320, 6, 35221]) cycle_id = 320
tensor_HBL-010RFC_RFS-DIG-201.pt data = torch.Size([320, 6, 35221]) cycle_id = 320
tensor_HBL-010RFC_RFS-DIG-301.pt data = torch.Size([320, 6, 35221]) cycle_id = 320
tensor_HBL-010RFC_RFS-DIG-401.pt data = torch.Size([331, 6, 35221]) cycle_id = 331


In [30]:
import math
import torch
import gc

# 前回確保したXがあればメモリを解放
if "X" in globals():
    del X
    gc.collect()

file_info = []

# 1回目：各ファイルの形状とcycle_idを調査
for path, digitizer in zip(files, digitizers):
    obj = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    data = obj["data"]
    cycle_id = obj["cycle_id"]

    if isinstance(cycle_id, torch.Tensor):
        cycle_ids = cycle_id.detach().cpu().tolist()
    else:
        cycle_ids = list(cycle_id)

    if data.shape[0] != len(cycle_ids):
        raise ValueError(
            f"{path.name}: "
            f"data={data.shape[0]}行、cycle_id={len(cycle_ids)}個"
        )

    if len(cycle_ids) != len(set(cycle_ids)):
        raise ValueError(
            f"{path.name}: cycle_idに重複があります"
        )

    feature_count = math.prod(data.shape[1:])

    file_info.append({
        "path": path,
        "digitizer": digitizer,
        "cycle_ids": cycle_ids,
        "data_shape": tuple(data.shape),
        "feature_count": feature_count,
        "dtype": data.dtype,
    })

    print(
        path.name,
        "shape =", tuple(data.shape),
        "1行にしたとき =", feature_count
    )

    del obj, data, cycle_id
    gc.collect()

tensor_HBL-010RFC_RFS-DIG-101.pt shape = (320, 6, 35221) 1行にしたとき = 211326
tensor_HBL-010RFC_RFS-DIG-201.pt shape = (320, 6, 35221) 1行にしたとき = 211326
tensor_HBL-010RFC_RFS-DIG-301.pt shape = (320, 6, 35221) 1行にしたとき = 211326
tensor_HBL-010RFC_RFS-DIG-401.pt shape = (331, 6, 35221) 1行にしたとき = 211326


In [31]:
common_set = set(file_info[0]["cycle_ids"])

for info in file_info[1:]:
    common_set &= set(info["cycle_ids"])

# 最初のファイルに登場する順序を維持
common_cycle_ids = [
    cycle_id
    for cycle_id in file_info[0]["cycle_ids"]
    if cycle_id in common_set
]

if len(common_cycle_ids) == 0:
    raise ValueError(
        "4ファイルに共通するcycle_idがありません"
    )

print("共通cycle数:", len(common_cycle_ids))

for info in file_info:
    print(
        info["digitizer"],
        "元のcycle数:",
        len(info["cycle_ids"]),
        "使用するcycle数:",
        len(common_cycle_ids)
    )

共通cycle数: 320
101 元のcycle数: 320 使用するcycle数: 320
201 元のcycle数: 320 使用するcycle数: 320
301 元のcycle数: 320 使用するcycle数: 320
401 元のcycle数: 331 使用するcycle数: 320


In [32]:
n_rows = len(common_cycle_ids)
total_features = sum(
    info["feature_count"]
    for info in file_info
)

X = torch.empty(
    (n_rows, total_features),
    dtype=file_info[0]["dtype"]
)

column_start = 0

for info in file_info:
    obj = torch.load(
        info["path"],
        map_location="cpu",
        weights_only=False
    )

    data = obj["data"]

    # cycle_idからファイル内の行番号を検索
    cycle_to_row = {
        cycle_id: row
        for row, cycle_id in enumerate(info["cycle_ids"])
    }

    row_indices = torch.tensor(
        [
            cycle_to_row[cycle_id]
            for cycle_id in common_cycle_ids
        ],
        dtype=torch.long
    )

    # 共通cycleだけを取り出して1行化
    flattened = data.index_select(
        0,
        row_indices
    ).reshape(n_rows, -1)

    column_end = column_start + flattened.shape[1]

    X[:, column_start:column_end].copy_(flattened)

    print(
        f"digitizer {info['digitizer']}:",
        tuple(data.shape),
        "→",
        tuple(flattened.shape),
        f"列 {column_start}:{column_end}"
    )

    column_start = column_end

    del obj, data, row_indices, flattened
    gc.collect()

digitizer 101: (320, 6, 35221) → (320, 211326) 列 0:211326
digitizer 201: (320, 6, 35221) → (320, 211326) 列 211326:422652
digitizer 301: (320, 6, 35221) → (320, 211326) 列 422652:633978
digitizer 401: (331, 6, 35221) → (320, 211326) 列 633978:845304


In [33]:
combined = {
    "data": X,
    "cycle_id": common_cycle_ids,
    "cavity_id": ["010RFC"] * len(common_cycle_ids),
}

print("最終データ:", combined["data"].shape)
print("cycle数:", len(combined["cycle_id"]))
print("1行の長さ:", combined["data"].shape[1])

最終データ: torch.Size([320, 845304])
cycle数: 320
1行の長さ: 845304


In [34]:
import torch
from torch.utils.data import DataLoader, Subset

print("正規化前:", X.shape)

row_mean = X.mean(dim=1, keepdim=True)
row_std = X.std(dim=1, keepdim=True).clamp_min(1e-6)

X.sub_(row_mean)
X.div_(row_std)

print("平均:", X.mean().item())
print("標準偏差:", X.std().item())

正規化前: torch.Size([320, 845304])
平均: -8.809007412402536e-10
標準偏差: 0.9999994039535522


全データ: 320
train: 256
validation: 32
test: 32


In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DownBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=9,
                stride=stride,
                padding=4
            ),
            nn.GroupNorm(
                min(8, out_channels),
                out_channels
            ),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=7,
                padding=3
            ),
            nn.GroupNorm(
                min(8, out_channels),
                out_channels
            ),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class CavityAutoencoder(nn.Module):
    def __init__(
        self,
        latent_dim=2,
        pooled_length=16
    ):
        super().__init__()

        self.latent_dim = latent_dim
        self.pooled_length = pooled_length

        # -------------------------
        # Encoder
        # -------------------------
        self.encoder = nn.ModuleList([
            DownBlock(1, 8, stride=8),
            DownBlock(8, 16, stride=4),
            DownBlock(16, 32, stride=4),
            DownBlock(32, 64, stride=4),
            DownBlock(64, 128, stride=4),
            DownBlock(128, 128, stride=2),
        ])

        # 入力の列数に依存せず固定長にする
        self.adaptive_pool = nn.AdaptiveAvgPool1d(
            pooled_length
        )

        # 最終的に2次元まで圧縮
        self.to_latent = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                128 * pooled_length,
                128
            ),
            nn.GELU(),

            nn.Linear(
                128,
                latent_dim
            )
        )

        # -------------------------
        # Decoderの開始部分
        # -------------------------
        self.from_latent = nn.Sequential(
            nn.Linear(
                latent_dim,
                128
            ),
            nn.GELU(),

            nn.Linear(
                128,
                128 * pooled_length
            ),
            nn.GELU()
        )

        # -------------------------
        # Decoder
        # -------------------------
        self.decoder = nn.ModuleList([
            UpBlock(128, 128),
            UpBlock(128, 64),
            UpBlock(64, 32),
            UpBlock(32, 16),
            UpBlock(16, 8),

            nn.Conv1d(
                8,
                1,
                kernel_size=7,
                padding=3
            )
        ])

    def encode(self, x):
        if x.ndim != 2:
            raise ValueError(
                "入力は [batch, features] です。"
                f"実際の形状: {x.shape}"
            )

        # [batch, features]
        # → [batch, 1, features]
        h = x.unsqueeze(1)

        original_sizes = []

        for block in self.encoder:
            original_sizes.append(
                h.shape[-1]
            )

            h = block(h)

        h = self.adaptive_pool(h)

        # [batch, 2]
        latent = self.to_latent(h)

        return latent, original_sizes

    def decode(
        self,
        latent,
        original_sizes
    ):
        batch_size = latent.shape[0]

        h = self.from_latent(latent)

        h = h.reshape(
            batch_size,
            128,
            self.pooled_length
        )

        for block, target_size in zip(
            self.decoder,
            reversed(original_sizes)
        ):
            h = F.interpolate(
                h,
                size=target_size,
                mode="linear",
                align_corners=False
            )

            h = block(h)

        # [batch, 1, features]
        # → [batch, features]
        reconstruction = h.squeeze(1)

        return reconstruction

    def forward(self, x):
        latent, original_sizes = self.encode(x)

        reconstruction = self.decode(
            latent,
            original_sizes
        )

        return reconstruction